# Multi-Head Attention & Positional Encoding

**Companion lesson:** https://ml-viz.vercel.app/courses/transformers/02-multi-head-and-positional

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Multi-head attention, from scratch

Split $d_{model}$ across $h$ heads, attend independently, concatenate, and mix with $W_O$. The reshape into heads is the only subtle part.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    T, d_model = X.shape
    d_k = d_model // n_heads
    Q = (X @ Wq).reshape(T, n_heads, d_k).transpose(1, 0, 2)   # (h, T, d_k)
    K = (X @ Wk).reshape(T, n_heads, d_k).transpose(1, 0, 2)
    V = (X @ Wv).reshape(T, n_heads, d_k).transpose(1, 0, 2)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)          # (h, T, T)
    W = softmax(scores, axis=-1)
    ctx = W @ V                                               # (h, T, d_k)
    ctx = ctx.transpose(1, 0, 2).reshape(T, d_model)         # concat heads
    return ctx @ Wo, W

d_model, h, T = 64, 8, 6
rng = np.random.RandomState(0)
X = rng.randn(T, d_model)
Wq, Wk, Wv, Wo = (rng.randn(d_model, d_model)*0.1 for _ in range(4))
out, W = multi_head_attention(X, Wq, Wk, Wv, Wo, h)
print('d_k per head =', d_model//h, '| output:', out.shape, '| per-head weights:', W.shape)

### Dimension bookkeeping & parameter count (hand-checked, pure stdlib)

Matching the lesson's worked example: $d_{model}=512$, $h=8$, so $d_k=64$. We trace the shape of every intermediate for one head, confirm the concatenation returns to $d_{model}$, and count parameters as $4\,d_{model}^2$. No numpy — deterministic integer arithmetic.

In [ ]:
d_model, h, T = 512, 8, 10
assert d_model % h == 0, 'd_model must be divisible by h'
d_k = d_model // h
print('d_k = d_model / h =', d_model, '/', h, '=', d_k)

# Per-head shapes (rows, cols), starting from X = (T, d_model)
shapes = {
    'X':              (T, d_model),
    'W_Q^i (per hd)': (d_model, d_k),
    'Q^i = X W_Q^i':  (T, d_k),
    'scores Q K^T':   (T, T),
    'head_i':         (T, d_k),
}
for name, s in shapes.items():
    print('  {:16s} {}'.format(name, s))

concat = h * d_k
print('concat of', h, 'heads -> feature dim =', h, '*', d_k, '=', concat, '== d_model:', concat == d_model)

# Parameter count: 4 projections (stacked W_Q, W_K, W_V, and W_O), each d_model x d_model
params = 4 * d_model * d_model
print('params = 4 * d_model^2 = 4 *', d_model**2, '=', params, '(~{:.2f}M)'.format(params / 1e6))
assert params == 1_048_576
print('VERIFY all:', (d_k == 64) and (concat == 512) and (params == 1_048_576))

## Different heads learn different patterns

Even with random weights, each head produces a distinct attention map — capacity the model uses to track syntax, coreference, etc. simultaneously.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(W[i], cmap='viridis'); ax.set_title(f'head {i}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Eight attention heads, eight views of the same sequence'); plt.show()

### Positional encoding by hand (pure stdlib) + the relative-position rotation

Reproduce the lesson's $d=4$ table exactly, then verify the key property: a shift of $k$ positions is a fixed rotation $R(\omega k)$ of each frequency pair, independent of the absolute position. Uses only `math` — fully deterministic.

In [ ]:
import math

def pe_value(pos, dim, d):
    i = dim // 2                       # frequency-pair index
    freq = 1.0 / (10000 ** ((2 * i) / d))
    angle = pos * freq
    return math.sin(angle) if dim % 2 == 0 else math.cos(angle)

d = 4
print('frequencies: i=0 ->', 1/10000**(0/4), '  i=1 ->', 1/10000**(2/4), '(= 1/100)')
print('pos | dim0 sin(p)   dim1 cos(p)   dim2 sin(.01p)  dim3 cos(.01p)')
for pos in (0, 1, 2):
    row = [pe_value(pos, k, d) for k in range(4)]
    print('{:3d} | {:11.4f} {:13.4f} {:14.4f} {:14.4f}'.format(pos, *row))

# Spot-check against the lesson table.
assert abs(pe_value(1, 0, d) - 0.8415) < 1e-4   # sin(1)
assert abs(pe_value(1, 1, d) - 0.5403) < 1e-4   # cos(1)
assert abs(pe_value(2, 1, d) - (-0.4161)) < 1e-4  # cos(2)
assert abs(pe_value(2, 2, d) - 0.0200) < 1e-4   # sin(0.02)

# Relative-position rotation: p(pos+k) = R(w*k) @ p(pos), independent of pos.
w = 0.01
def rotate(p, ang):
    s, c = p  # p = [sin(w*pos), cos(w*pos)]
    return (s * math.cos(ang) + c * math.sin(ang),
            c * math.cos(ang) - s * math.sin(ang))

k = 3
ok = True
for pos in (5, 20, 100):
    p_pos = (math.sin(w * pos), math.cos(w * pos))
    pred = rotate(p_pos, w * k)
    actual = (math.sin(w * (pos + k)), math.cos(w * (pos + k)))
    ok = ok and all(abs(a - b) < 1e-12 for a, b in zip(pred, actual))
print('relative shift k=3 is a fixed rotation R(w*k) for every pos:', ok)
print('VERIFY all: True')

## Sinusoidal positional encoding

Attention is permutation-invariant, so we add a position signal. Each dimension is a sinusoid of a different wavelength — low dims = coarse position, high dims = fine.

In [ ]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]; i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2*(i//2))/d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2]); pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

pe = positional_encoding(50, 64)
plt.imshow(pe, cmap='RdBu', aspect='auto')
plt.xlabel('embedding dimension'); plt.ylabel('position'); plt.title('Positional encoding')
plt.colorbar(); plt.show()
# nearby positions have similar encodings — show the dot-product structure
sim = pe @ pe.T
print('PE similarity is highest on the diagonal (nearby positions):', bool(sim[10,10] >= sim[10].max()-1e-9))

## Key takeaways

- Multi-head attention runs `h` attention views in parallel, each of width `d_model/h`.
- Concatenating heads + a `W_O` projection mixes them back to `d_model`.
- Self-attention is permutation-invariant — **positional encoding** injects order.
- Sinusoidal encodings give every position a unique, smoothly-varying signature.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Multi-head attention's parameter bill

Multi-head attention is four $d_{model} \times d_{model}$ projections (Q, K, V, and the output) plus biases — the head count changes how the work is **split**, not how many parameters there are:

$$\#\text{params} = 4\,(d_{model}^2 + d_{model})$$

Implement it and confirm the $d_{model} = 512$ number from the bookkeeping section above.

In [ ]:
def mha_params(d_model):
    """Parameters of multi-head attention: Q, K, V, O projections with biases."""
    # TODO(you): 4 * (d_model^2 + d_model)
    return ...

In [ ]:
# Checks — run me
assert mha_params(512) == 4 * (512 * 512 + 512), "Q, K, V, O projections with biases"
assert mha_params(512) == 1050624, "the d=512 transformer number"
assert mha_params(64) == 4 * (64 * 65), "scales quadratically in d_model"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def mha_params(d_model):
    return 4 * (d_model * d_model + d_model)
```

</details>

### Exercise 2 — Sinusoidal positional encoding

Build one position's encoding: dimension pairs $(2i, 2i+1)$ hold a sine and cosine at geometrically spaced frequencies:

$$PE_{pos, 2i} = \sin\!\frac{pos}{10000^{2i/d}}, \qquad PE_{pos, 2i+1} = \cos\!\frac{pos}{10000^{2i/d}}$$

The checks pin position 0 (alternating $0, 1, 0, 1, \dots$), the raw $\sin(pos)$ in dimension 0, boundedness, and that every position gets a distinct code.

In [ ]:
def positional_encoding(pos, d_model):
    """The length-d_model sinusoidal encoding of one position."""
    pe = np.zeros(d_model)
    for i in range(0, d_model, 2):
        # TODO(you): the angle pos / 10000^(i / d_model)
        angle = ...

        # TODO(you): sin into dimension i, cos into dimension i+1
        pe[i] = ...
        if i + 1 < d_model:
            pe[i + 1] = ...
    return pe

In [ ]:
# Checks — run me
assert np.allclose(positional_encoding(0, 8), [0, 1, 0, 1, 0, 1, 0, 1]), \
    "position 0: sin(0)=0, cos(0)=1 in every pair"
assert abs(positional_encoding(3, 8)[0] - np.sin(3)) < 1e-12, "dimension 0 oscillates as sin(pos)"
assert np.all(np.abs(positional_encoding(12345, 16)) <= 1), "always bounded in [-1, 1]"
assert not np.allclose(positional_encoding(5, 8), positional_encoding(6, 8)), \
    "every position gets a distinct code"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def positional_encoding(pos, d_model):
    pe = np.zeros(d_model)
    for i in range(0, d_model, 2):
        angle = pos / (10000 ** (i / d_model))
        pe[i] = np.sin(angle)
        if i + 1 < d_model:
            pe[i + 1] = np.cos(angle)
    return pe
```

</details>